# Install

In [8]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

#  Import

In [9]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import io
from datetime import datetime, timezone
from azure.storage.blob import BlobServiceClient

# Config

In [10]:
APP_TOKEN = os.environ["SOCRATA_APP_TOKEN"]
CONN_STR = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")

API_URL = "https://data.cityofnewyork.us/resource/i4gi-tjb9.json"
LIMIT = 50000  # max per request

client = BlobServiceClient.from_connection_string(CONN_STR)
print("Config xong")

Config xong


# Hàm pull 1 tháng

In [11]:
def pull_month(year, month):
    start = f"{year}-{month:02d}-01T00:00:00"
    if month == 12:
        end = f"{year+1}-01-01T00:00:00"
    else:
        end = f"{year}-{month+1:02d}-01T00:00:00"
    
    all_rows = []
    offset = 0
    
    while True:
        params = {
            "$where": f"data_as_of >= '{start}' AND data_as_of < '{end}'",
            "$limit": LIMIT,
            "$offset": offset,
            "$order": "data_as_of ASC",
            "$$app_token": APP_TOKEN
        }
        resp = requests.get(API_URL, params=params, timeout=60)
        resp.raise_for_status()
        rows = resp.json()
        
        if not rows:
            break
            
        all_rows.extend(rows)
        offset += LIMIT
        print(f"  {year}-{month:02d}: pulled {len(all_rows)} rows...", end="\r")
        
        if len(rows) < LIMIT:
            break
    
    return all_rows

# Hàm upload parquet lên Azure

In [12]:
def upload_parquet(df, year, month):
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    
    table = pa.Table.from_pandas(df, preserve_index=False)
    buf = io.BytesIO()
    pq.write_table(table, buf)
    buf.seek(0)
    
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    blob.upload_blob(buf, overwrite=True)
    print(f"Uploaded: {blob_path} ({len(df)} rows)")

# Pull toàn bộ 01/2024 đến nay

In [ ]:
from datetime import date

now = date.today()
months = []
y, m = 2024, 1
while (y, m) <= (now.year, now.month):
    months.append((y, m))
    m += 1
    if m > 12:
        m = 1
        y += 1

print(f"Tổng số tháng: {len(months)}")

def is_already_uploaded(year, month):
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    try:
        blob.get_blob_properties()
        return True
    except:
        return False

for year, month in months:
    if is_already_uploaded(year, month):
        print(f"Skip {year}-{month:02d} (đã có)")
        continue
        
    print(f"\nPulling {year}-{month:02d}...")
    rows = pull_month(year, month)
    
    if not rows:
        print(f"Không có data {year}-{month:02d}, bỏ qua")
        continue
    
    df = pd.DataFrame(rows)
    upload_parquet(df, year, month)

print("\nHoàn thành pull Bronze historical")

Tổng số tháng: 30
Skip 2024-01 (đã có)

Pulling 2024-02...
  2024-02: pulled 450000 rows...

# Verify

In [ ]:
# Kiểm tra vài tháng đầu
for year, month in months[:3]:
    blob_path = f"bronze/dot_traffic_speeds/historical/year={year}/month={month:02d}/raw_{year}_{month:02d}.parquet"
    blob = client.get_blob_client(container=CONTAINER, blob=blob_path)
    props = blob.get_blob_properties()
    print(f"{blob_path}: {props.size:,} bytes")